In [1]:
 """SESSION_33: SOLOv2 ResNet-50-FPN instance segmentation.

Datasets supported (auto-detected on Kaggle, or pass --data-root):
  SkinDisNet_2_Annotation   manifest-based source-grouped CV; test split locked.
                            With SkinDisNet_Metadata.csv attached, groups by
                            Patient_id instead of source image.
  skin_disease_yolo         filename-based source-grouped CV (Roboflow .rf. stems).
                            All splits are POOLED because source variants span
                            train/test. The original test split is NOT independent.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import os
import re
import shutil
import statistics
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path
from typing import Any


PYTHON_ENV = Path("/kaggle/working/session32_33_mmdet_py310")
PYTHON_BIN = PYTHON_ENV / "bin" / "python"
TORCH_INDEX = "https://download.pytorch.org/whl/cu113"
MMCV_INDEX = "https://download.openmmlab.com/mmcv/dist/cu113/torch1.12.0/index.html"
MMDET_SOURCE = Path("/kaggle/working/mmdetection_v2.28.2")
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_NAMES = {
    "train": ("train",),
    "valid": ("valid", "val", "validation"),
    "test": ("test",),
}
# Roboflow augmentation prefixes that precede the source stem (skin_disease_yolo)
AUGMENT_PREFIX = re.compile(
    r"^(?:"
    r"Crop-(?:100450_100400|100450_0300|100350_100300|0350_100400|0350_0300)|"
    r"padd-(?:100100_100100|1000_00|0100_00|00_1000|00_0100)|"
    r"Resize-450_400|"
    r"flip-(?:-1|0|1)|"
    r"(?:hue|saturation)-(?:200|150|100|50)|"
    r"superpixels-(?:200|100|75|50|25)"
    r")"
)
SKIN_DISEASE_YOLO_CLASSES = [
    "acne", "Darier_s-Disease", "Epidermolysis-Bullosa-Pruriginosa",
    "Hailey-Hailey-Disease", "Hemangiome", "Impetigo", "Leishmanios",
    "Lupus-Erythematosus-Chronicus-Discoides", "Melanoma",
    "Molluscum-Contagiosum", "Porokeratosis", "Psoriasis", "Tinea-Corporis",
    "Tungiasis", "basal-cell-carcinoma", "eczema", "lichen", "nevus",
    "normal skin",
]


def _run(command: list[str], *, env: dict[str, str] | None = None) -> None:
    print("+", " ".join(command), flush=True)
    subprocess.run(command, check=True, env=env)


def _install_environment(python: Path) -> None:
    probe = [
        str(python), "-c",
        "import cv2, pkg_resources, torch, torchvision, mmcv, mmdet; "
        "from importlib.metadata import version; "
        "from mmcv.ops import nms; "
        "assert torch.cuda.is_available(), 'CUDA is unavailable'; "
        "assert mmcv.__version__ == '1.7.2'; "
        "assert mmdet.__version__ == '2.28.2'; "
        "assert version('setuptools') == '69.5.1'; "
        "print(torch.__version__, torchvision.__version__, "
        "mmcv.__version__, mmdet.__version__, "
        "'setuptools=' + version('setuptools'))",
    ]
    check = subprocess.run(probe, text=True, capture_output=True)
    if check.returncode == 0:
        print("[environment] verified:", check.stdout.strip(), flush=True)
        return
    print("[environment] installing the pinned CUDA 11.3 stack", flush=True)

    installer = [str(python), "-m", "pip", "install", "-q"]
    _run([
        *installer, "--upgrade", "--force-reinstall", "--no-cache-dir",
        "setuptools==69.5.1", "wheel==0.44.0",
    ])
    _run([
        *installer, "--upgrade", "--no-cache-dir",
        "opencv-python==4.8.1.78",
    ])
    _run([*installer, "numpy==1.23.5"])
    _run([
        *installer, "torch==1.12.0+cu113", "torchvision==0.13.0+cu113",
        "--index-url", TORCH_INDEX, "--extra-index-url", "https://pypi.org/simple",
    ])
    _run([
        *installer, "--no-deps", "--find-links", MMCV_INDEX,
        "mmcv-full==1.7.2",
    ])
    _run([
        *installer, "--no-deps", "mmdet==2.28.2",
    ])
    _run([
        *installer,
        "addict", "matplotlib==3.7.5", "opencv-python==4.8.1.78",
        "packaging", "Pillow", "pycocotools", "PyYAML", "rich",
        "scipy==1.10.1", "six", "terminaltables", "tqdm", "yapf==0.40.1",
    ])

    check = subprocess.run(probe, text=True, capture_output=True)
    if check.returncode:
        raise RuntimeError(
            "Pinned MMDetection environment failed its CUDA/MMCV import check:\n"
            + (check.stdout + check.stderr)[-5000:]
        )
    print("[environment] verified:", check.stdout.strip(), flush=True)


def _is_runtime_source(source: str, session: str) -> bool:
    return (
        "def run_session(" in source
        and "def _bootstrap(" in source
        and session in source
    )


def _notebook_runtime_source(session: str) -> str:
    try:
        from IPython import get_ipython
    except ImportError as exc:
        raise RuntimeError(
            f"Cannot locate the {session} source script for the Python 3.10 restart."
        ) from exc

    shell = get_ipython()
    if shell is None:
        raise RuntimeError(
            f"Cannot locate the {session} source script for the Python 3.10 restart."
        )
    history = shell.user_ns.get("In", [])
    for source in reversed(history):
        if isinstance(source, str) and _is_runtime_source(source, session):
            return source
    raise RuntimeError(
        f"Could not find the {session} script in the current notebook cell history. "
        "Run the complete Session 32 script in one cell."
    )


def _filter_notebook_args(arguments: list[str]) -> list[str]:
    filtered: list[str] = []
    ignored: list[str] = []
    skip_next = False
    for argument in arguments:
        if skip_next:
            ignored.append(argument)
            skip_next = False
            continue
        if argument == "-f":
            ignored.append(argument)
            skip_next = True
            continue
        if argument.startswith("-f="):
            ignored.append(argument)
            continue
        if argument.startswith("--HistoryManager."):
            ignored.append(argument)
            continue
        filtered.append(argument)
    if ignored:
        print(f"[args] ignored Jupyter kernel arguments: {ignored}", flush=True)
    return filtered


def _bootstrap(entrypoint: Path, argv: list[str], session: str) -> int | None:
    argv = _filter_notebook_args(argv)
    if sys.version_info[:2] == (3, 10):
        return None
    working = Path("/kaggle/working")
    if not working.is_dir():
        raise RuntimeError(
            "These sessions bootstrap Python 3.10 for the pinned YOLACT/SOLOv2 "
            "MMDetection stack and must run in a Kaggle notebook."
        )
    uv = [sys.executable, "-m", "uv"]
    if subprocess.run(
        [*uv, "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    ).returncode:
        _run([sys.executable, "-m", "pip", "install", "-q", "uv"])
    if not PYTHON_BIN.is_file():
        _run([*uv, "python", "install", "3.10"])
        _run([
            *uv, "venv", "--seed", "--python", "3.10", str(PYTHON_ENV),
        ])

    source_dir = working / "session32_33_sources"
    source_dir.mkdir(parents=True, exist_ok=True)
    if entrypoint.is_file():
        source = entrypoint.read_text(encoding="utf-8")
    else:
        source = ""
    if _is_runtime_source(source, session):
        copied_script = source_dir / entrypoint.name
        shutil.copy2(entrypoint, copied_script)
    else:
        source = _notebook_runtime_source(session)
        copied_script = source_dir / f"{session.lower()}_runtime.py"
        copied_script.write_text(source, encoding="utf-8")
    _install_environment(PYTHON_BIN)
    print(f"[python] restarting under Python 3.10 for {session}", flush=True)
    return subprocess.call(
        [str(PYTHON_BIN), "-u", str(copied_script), *argv],
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )


def _find_data_root(explicit: str | None) -> Path:
    if explicit:
        root = Path(explicit).expanduser().resolve()
        if root.is_file() and root.name.lower() == "data.yaml":
            return root
        if not root.is_dir():
            raise FileNotFoundError(f"Dataset root does not exist: {root}")
        return root

    input_root = Path("/kaggle/input")
    candidates: set[Path] = set()
    if input_root.is_dir():
        for yaml_file in input_root.rglob("data.yaml"):
            candidates.add(yaml_file.parent.resolve())
        for annotation in input_root.rglob("_annotations.coco.json"):
            if annotation.parent.name.lower() in SPLIT_NAMES["train"]:
                candidates.add(annotation.parent.parent.resolve())
    manifest_candidates = []
    skin_disease_yolo_candidates = []
    for candidate in candidates:
        if candidate.name.lower().replace("-", "_") == "skin_disease_yolo":
            skin_disease_yolo_candidates.append(candidate)
            continue
        manifest = candidate / "manifest.csv"
        if not manifest.is_file():
            continue
        with manifest.open(newline="", encoding="utf-8") as handle:
            columns = set(next(csv.reader(handle), []))
        if {"name", "split", "source", "augmented"} <= columns:
            manifest_candidates.append(candidate)
    if len(manifest_candidates) == 1 and not skin_disease_yolo_candidates:
        return manifest_candidates[0]
    if len(skin_disease_yolo_candidates) == 1 and not manifest_candidates:
        return skin_disease_yolo_candidates[0]
    if len(manifest_candidates) + len(skin_disease_yolo_candidates) == 1:
        return (manifest_candidates or skin_disease_yolo_candidates)[0]
    if len(candidates) == 1:
        return next(iter(candidates))
    raise ValueError(
        f"Found candidate datasets under {input_root}: "
        f"{sorted(map(str, candidates))}. Pass --data-root explicitly."
    )


def _load_yaml(path: Path) -> dict[str, Any]:
    import yaml

    loaded = yaml.safe_load(path.read_text(encoding="utf-8"))
    if not isinstance(loaded, dict):
        raise ValueError(f"Expected a mapping in {path}")
    return loaded


def _names_from_yaml(data: dict[str, Any]) -> list[str]:
    names = data.get("names")
    if isinstance(names, dict):
        names = [names[key] for key in sorted(names, key=lambda value: int(value))]
    if not isinstance(names, list) or not names:
        raise ValueError("YOLO data.yaml must define a non-empty names list.")
    return [str(name) for name in names]


def _yolo_image_dir(yaml_path: Path, data: dict[str, Any], split: str) -> Path:
    split_keys = SPLIT_NAMES[split]
    root_value = data.get("path")
    root = Path(root_value).expanduser() if root_value else yaml_path.parent
    if not root.is_absolute():
        root = (yaml_path.parent / root).resolve()
    if not root.is_dir():
        root = yaml_path.parent
    for key in split_keys:
        split_value = data.get(key)
        if not split_value:
            continue
        path = Path(str(split_value)).expanduser()
        path = path if path.is_absolute() else (root / path).resolve()
        if path.is_dir():
            return path
    for name in split_keys:
        for candidate in (root / name / "images", root / name):
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError(f"Could not locate the {split} image directory in {yaml_path}")


def _polygon_area(xs: list[float], ys: list[float]) -> float:
    return abs(sum(
        xs[index] * ys[(index + 1) % len(xs)]
        - xs[(index + 1) % len(xs)] * ys[index]
        for index in range(len(xs))
    )) / 2


def _source_manifest(yaml_path: Path) -> dict[tuple[str, str], str]:
    manifest_path = yaml_path.parent / "manifest.csv"
    if not manifest_path.is_file():
        return {}
    source_groups: dict[tuple[str, str], str] = {}
    with manifest_path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        required = {"name", "split", "source", "augmented"}
        if not required <= set(reader.fieldnames or []):
            raise ValueError(
                f"{manifest_path} must contain columns {sorted(required)}"
            )
        for row in reader:
            split = row.get("split", "").strip().lower()
            split = "valid" if split in {"val", "validation"} else split
            key = (split, Path(row.get("name", "")).stem)
            source = row.get("source", "").strip()
            if split not in SPLIT_NAMES or not key[1] or not source or key in source_groups:
                raise ValueError(f"Invalid or duplicate entry in {manifest_path}: {row}")
            source_groups[key] = source
    if not source_groups:
        raise ValueError(f"No source entries found in {manifest_path}")

    # Patient grouping: when SkinDisNet_Metadata.csv is attached, remap every source id
    # to its Patient_id so CV folds are grouped by patient, not by source image.
    candidates = list(yaml_path.parent.parent.rglob("SkinDisNet_Metadata.csv"))
    if Path("/kaggle/input").is_dir():
        candidates += list(Path("/kaggle/input").rglob("SkinDisNet_Metadata.csv"))
    for meta_path in candidates:
        patient_map: dict[str, str] = {}
        with meta_path.open(newline="", encoding="utf-8-sig") as stream:
            for row in csv.DictReader(stream):
                img_id = re.sub(
                    r"\s*\((\d+)\)\s*$", r"_\1", row.get("Image_id", "").strip()
                )
                if img_id and row.get("Patient_id", "").strip():
                    patient_map[img_id] = row["Patient_id"].strip()
        mapped = {v for v in source_groups.values() if v in patient_map}
        if mapped:
            source_groups = {
                k: patient_map.get(v, v) for k, v in source_groups.items()
            }
            patients = {patient_map[v] for v in mapped}
            print(
                f"  [data] patient grouping: {len(mapped)} sources mapped to "
                f"{len(patients)} Patient_ids from {meta_path}",
                flush=True,
            )
            break

    return source_groups


def _link_image(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    source = source.resolve()
    if destination.is_symlink():
        if destination.resolve() == source:
            return
        destination.unlink()
    elif destination.exists():
        raise FileExistsError(
            f"Refusing to replace non-symlink staged image: {destination}"
        )
    destination.symlink_to(source)


def _yolo_split(
    yaml_path: Path,
    split: str,
    destination: Path,
    staged_images: Path,
    source_groups: dict[tuple[str, str], str] | None = None,
) -> tuple[list[str], dict[str, Any]]:
    from PIL import Image

    data = _load_yaml(yaml_path)
    names = _names_from_yaml(data)
    image_dir = _yolo_image_dir(yaml_path, data, split)
    label_dir = image_dir.parent / "labels"
    image_paths = sorted(
        path for path in image_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )
    if not image_paths:
        raise FileNotFoundError(f"No images found in {image_dir}")
    staged_images.mkdir(parents=True, exist_ok=True)
    images: list[dict[str, Any]] = []
    annotations: list[dict[str, Any]] = []
    counts: Counter[str] = Counter()
    next_annotation_id = 1
    skipped = 0
    seen_source_keys: set[tuple[str, str]] = set()

    for image_id, source in enumerate(image_paths, 1):
        with Image.open(source) as image:
            width, height = image.size
        staged_name = f"{image_id:08d}{source.suffix.lower()}"
        _link_image(source, staged_images / staged_name)
        image_record = {
            "id": image_id,
            "file_name": staged_name,
            "width": width,
            "height": height,
        }
        if source_groups:
            source_key = (split, source.stem)
            source_group = source_groups.get(source_key)
            if source_group is None:
                raise ValueError(
                    f"Image {source} has no source group mapping."
                )
            image_record["source_group"] = source_group
            seen_source_keys.add(source_key)
        images.append(image_record)
        labels = label_dir / f"{source.stem}.txt"
        if not labels.is_file():
            continue
        for line in labels.read_text(encoding="utf-8").splitlines():
            fields = line.split()
            if len(fields) < 7 or (len(fields) - 1) % 2:
                skipped += 1
                continue
            try:
                class_id = int(fields[0])
                coords = [min(1.0, max(0.0, float(value))) for value in fields[1:]]
            except ValueError:
                skipped += 1
                continue
            if not 0 <= class_id < len(names):
                skipped += 1
                continue
            xs = [coords[index] * width for index in range(0, len(coords), 2)]
            ys = [coords[index] * height for index in range(1, len(coords), 2)]
            area = _polygon_area(xs, ys)
            x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
            if area < 1 or x1 <= x0 or y1 <= y0:
                skipped += 1
                continue
            annotations.append({
                "id": next_annotation_id,
                "image_id": image_id,
                "category_id": class_id + 1,
                "segmentation": [[value for pair in zip(xs, ys) for value in pair]],
                "area": area,
                "bbox": [x0, y0, x1 - x0, y1 - y0],
                "iscrowd": 0,
            })
            counts[names[class_id]] += 1
            next_annotation_id += 1

    if source_groups:
        expected = {key for key in source_groups if key[0] == split}
        missing_images = expected - seen_source_keys
        if missing_images:
            raise ValueError(
                f"{len(missing_images)} source group entries have no {split} image; "
                f"first missing entry: {sorted(missing_images)[0]}"
            )

    coco = {
        "images": images,
        "annotations": annotations,
        "categories": [
            {"id": index + 1, "name": name, "supercategory": "skin"}
            for index, name in enumerate(names)
        ],
    }
    destination.write_text(json.dumps(coco), encoding="utf-8")
    print(
        f"[{split}] YOLO polygons -> COCO: {len(images)} images, "
        f"{len(annotations)} instances, skipped malformed labels={skipped}"
    )
    return names, coco


def _coco_split(
    root: Path,
    split: str,
    destination: Path,
    staged_images: Path,
) -> tuple[list[str], dict[str, Any]]:
    from PIL import Image

    split_dir = next(
        (root / alias for alias in SPLIT_NAMES[split]
         if (root / alias).is_dir()),
        None,
    )
    if split_dir is None:
        raise FileNotFoundError(f"Missing {split} directory under {root}")
    source_json = next(
        (split_dir / name for name in (
            "_annotations.coco.json", "annotations.json", "instances.json"
        ) if (split_dir / name).is_file()),
        None,
    )
    if source_json is None:
        raise FileNotFoundError(f"Missing COCO annotation JSON in {split_dir}")

    raw = json.loads(source_json.read_text(encoding="utf-8"))
    source_categories = raw.get("categories", [])
    class_names: list[str] = []
    source_to_class: dict[int, int] = {}
    dedupe: dict[str, int] = {}
    for category in sorted(source_categories, key=lambda item: int(item["id"])):
        name = str(category["name"]).strip()
        normalized = " ".join(name.casefold().split())
        if not name:
            raise ValueError(f"Empty category name in {source_json}")
        class_index = dedupe.get(normalized)
        if class_index is None:
            class_index = len(class_names)
            class_names.append(name)
            dedupe[normalized] = class_index
        source_to_class[int(category["id"])] = class_index
    if not class_names:
        raise ValueError(f"No categories declared in {source_json}")

    def image_index() -> dict[str, Path]:
        files = (
            path for path in split_dir.rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
        )
        index: dict[str, Path] = {}
        for path in files:
            index.setdefault(path.name, path)
        return index

    by_name = image_index()
    staged_images.mkdir(parents=True, exist_ok=True)
    remapped_images: list[dict[str, Any]] = []
    old_to_new: dict[int, int] = {}
    for image_id, image in enumerate(raw.get("images", []), 1):
        old_id = int(image["id"])
        file_name = str(image["file_name"])
        source = (split_dir / file_name).resolve()
        if not source.is_file():
            source = by_name.get(Path(file_name).name)
        if source is None or not source.is_file():
            raise FileNotFoundError(f"COCO image {file_name!r} is missing from {split_dir}")
        with Image.open(source) as handle:
            width, height = handle.size
        if int(image.get("width", width)) != width or int(image.get("height", height)) != height:
            raise ValueError(f"Image dimensions disagree with COCO metadata: {source}")
        staged_name = f"{image_id:08d}{source.suffix.lower()}"
        _link_image(source, staged_images / staged_name)
        old_to_new[old_id] = image_id
        remapped_images.append({
            "id": image_id,
            "file_name": staged_name,
            "width": width,
            "height": height,
        })

    remapped_annotations: list[dict[str, Any]] = []
    for annotation_id, annotation in enumerate(raw.get("annotations", []), 1):
        old_category = int(annotation["category_id"])
        if old_category not in source_to_class:
            raise ValueError(
                f"Unknown category_id={old_category} in {source_json}"
            )
        old_image = int(annotation["image_id"])
        if old_image not in old_to_new:
            raise ValueError(f"Annotation references missing image ID {old_image}")
        if not annotation.get("segmentation"):
            continue
        converted = dict(annotation)
        converted.update({
            "id": annotation_id,
            "image_id": old_to_new[old_image],
            "category_id": source_to_class[old_category] + 1,
            "iscrowd": int(annotation.get("iscrowd", 0)),
        })
        remapped_annotations.append(converted)

    coco = {
        "images": remapped_images,
        "annotations": remapped_annotations,
        "categories": [
            {"id": index + 1, "name": name, "supercategory": "skin"}
            for index, name in enumerate(class_names)
        ],
    }
    destination.write_text(json.dumps(coco), encoding="utf-8")
    print(
        f"[{split}] COCO annotations: {len(remapped_images)} images, "
        f"{len(remapped_annotations)} instance masks; classes={len(class_names)}"
    )
    return class_names, coco


def _filename_source_groups(
    yaml_path: Path, split: str
) -> dict[tuple[str, str], str]:
    """Build source_groups dict from Roboflow .rf. filename stems (no manifest needed)."""
    data = _load_yaml(yaml_path)
    image_dir = _yolo_image_dir(yaml_path, data, split)
    groups: dict[tuple[str, str], str] = {}
    for img in image_dir.iterdir():
        if img.is_file() and img.suffix.lower() in IMAGE_SUFFIXES:
            raw_stem = img.stem.split(".rf.", 1)[0]
            source = AUGMENT_PREFIX.sub("", raw_stem, count=1)
            groups[(split, img.stem)] = source
    return groups


def prepare_dataset(
    root: Path, work_dir: Path, pool_all_splits: bool = False
) -> tuple[list[str], dict[str, Path]]:
    yaml_path = root if root.is_file() else next(
        (path for path in (root / "data.yaml", root / "data.yml") if path.is_file()),
        None,
    )
    if yaml_path is None and root.is_dir():
        yaml_candidates = sorted(root.rglob("data.yaml"))
        if len(yaml_candidates) == 1:
            yaml_path = yaml_candidates[0]

    source_groups: dict[tuple[str, str], str] = {}
    if pool_all_splits and yaml_path is not None:
        for sp in ("train", "valid", "test"):
            source_groups.update(_filename_source_groups(yaml_path, sp))
    elif yaml_path is not None:
        source_groups = _source_manifest(yaml_path)

    coco_dir = work_dir / "coco"
    image_root = work_dir / "images"
    coco_dir.mkdir(parents=True, exist_ok=True)
    common_classes: list[str] | None = None
    annotations: dict[str, Path] = {}
    for split in ("train", "valid", "test"):
        output = coco_dir / f"{split}.json"
        staged = image_root / split
        split_groups = (
            {k: v for k, v in source_groups.items() if k[0] == split}
            if source_groups else {}
        )
        if yaml_path is not None:
            names, coco = _yolo_split(
                yaml_path, split, output, staged, split_groups or None
            )
        else:
            names, coco = _coco_split(root, split, output, staged)
        if common_classes is None:
            common_classes = names
        elif names != common_classes:
            raise ValueError(
                f"Class list for {split} differs from train: {names} != {common_classes}"
            )
        if not coco["images"]:
            raise ValueError(f"The {split} split has no images.")
        if split != "train" and not coco["annotations"]:
            raise ValueError(f"The {split} split has no instance masks.")
        annotations[split] = output
    assert common_classes is not None
    return common_classes, annotations


def _make_development_coco(
    annotations: dict[str, Path], destination: Path, image_root: Path,
    pool_test: bool = False,
) -> dict[str, Any]:
    """Merge train/valid (and optionally test) into a single development COCO dataset."""
    merged_images: list[dict[str, Any]] = []
    merged_annotations: list[dict[str, Any]] = []
    categories: list[dict[str, Any]] | None = None
    next_image_id = 1
    next_annotation_id = 1
    destination_images = image_root / "development"

    splits_to_pool = ("train", "valid", "test") if pool_test else ("train", "valid")
    for split in splits_to_pool:
        dataset = json.loads(annotations[split].read_text(encoding="utf-8"))
        current_categories = dataset["categories"]
        if categories is None:
            categories = current_categories
        elif current_categories != categories:
            raise ValueError(f"Category definitions differ in the {split} split.")
        old_to_new: dict[int, int] = {}
        for image in dataset["images"]:
            source = image_root / split / image["file_name"]
            if not source.is_file():
                raise FileNotFoundError(f"Prepared image missing: {source}")
            new_id = next_image_id
            next_image_id += 1
            old_to_new[int(image["id"])] = new_id
            staged_name = f"{split}_{image['file_name']}"
            staged_path = destination_images / staged_name
            _link_image(source, staged_path)
            merged_images.append({
                **image,
                "id": new_id,
                "file_name": str(staged_path.resolve()),
            })
        for annotation in dataset["annotations"]:
            merged_annotations.append({
                **annotation,
                "id": next_annotation_id,
                "image_id": old_to_new[int(annotation["image_id"])],
            })
            next_annotation_id += 1

    if categories is None:
        raise ValueError("No categories found while combining development splits.")
    merged = {
        "images": merged_images,
        "annotations": merged_annotations,
        "categories": categories,
    }
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(merged), encoding="utf-8")
    return merged


def _assert_test_source_isolation(
    annotations: dict[str, Path], skip: bool = False
) -> int:
    source_groups: dict[str, set[str]] = {}
    for split in ("train", "valid", "test"):
        document = json.loads(annotations[split].read_text(encoding="utf-8"))
        source_groups[split] = {
            str(image["source_group"])
            for image in document["images"]
            if "source_group" in image
        }
    development = source_groups["train"] | source_groups["valid"]
    if not skip:
        overlap = development & source_groups["test"]
        if overlap:
            raise ValueError(
                "Source-manifest groups overlap the locked test split; refusing "
                f"cross-split evaluation. First overlapping source: {sorted(overlap)[0]}"
            )
    return len(development | source_groups["test"])


def _stratified_folds(dataset: dict[str, Any], fold_count: int, seed: int) -> list[list[int]]:
    """Balance folds by instance counts while keeping source variants together."""
    import random

    image_ids = [int(image["id"]) for image in dataset["images"]]
    categories = [int(category["id"]) for category in dataset["categories"]]
    image_labels: dict[int, Counter[int]] = {image_id: Counter() for image_id in image_ids}
    for annotation in dataset["annotations"]:
        image_labels[int(annotation["image_id"])][int(annotation["category_id"])] += 1
    members_by_group: dict[str, list[int]] = {}
    for image in dataset["images"]:
        image_id = int(image["id"])
        source = image.get("source_group")
        group = f"source:{source}" if source is not None else f"image:{image_id}"
        members_by_group.setdefault(group, []).append(image_id)
    group_labels = {
        group: sum((image_labels[image_id] for image_id in members), Counter())
        for group, members in members_by_group.items()
    }
    rng = random.Random(seed)
    tie_break = {group: rng.random() for group in members_by_group}
    rarity = {
        category_id: sum(
            count > 0 for counts in group_labels.values()
            for label, count in counts.items() if label == category_id
        )
        for category_id in categories
    }
    ordered = sorted(
        members_by_group,
        key=lambda group: (
            min(
                (rarity[label] for label in group_labels[group]),
                default=len(members_by_group),
            ),
            -sum(group_labels[group].values()),
            tie_break[group],
        ),
    )
    targets = {
        category_id: sum(
            counts[category_id] for counts in image_labels.values()
        ) / fold_count
        for category_id in categories
    }
    folds: list[list[int]] = [[] for _ in range(fold_count)]
    fold_counts = [Counter() for _ in range(fold_count)]
    target_images = len(image_ids) / fold_count
    fold_sizes = [0] * fold_count

    for group in ordered:
        labels = group_labels[group]
        group_size = len(members_by_group[group])
        def score(fold_index: int) -> tuple[float, float, float]:
            label_cost = sum(
                ((fold_counts[fold_index][label] + count - targets[label])
                 / max(1.0, targets[label])) ** 2
                - ((fold_counts[fold_index][label] - targets[label])
                   / max(1.0, targets[label])) ** 2
                for label, count in labels.items()
            )
            size_cost = (
                (fold_sizes[fold_index] + group_size - target_images) ** 2
                - (fold_sizes[fold_index] - target_images) ** 2
            ) / max(1.0, target_images**2)
            return label_cost, size_cost, rng.random()

        selected = min(range(fold_count), key=score)
        folds[selected].extend(members_by_group[group])
        fold_counts[selected].update(labels)
        fold_sizes[selected] += group_size

    if any(not fold for fold in folds):
        raise RuntimeError("Stratified fold assignment produced an empty fold.")
    return folds


def _assert_fold_source_isolation(
    dataset: dict[str, Any], folds: list[list[int]]
) -> None:
    fold_by_image = {
        image_id: fold_index
        for fold_index, image_ids in enumerate(folds)
        for image_id in image_ids
    }
    group_folds: dict[str, set[int]] = {}
    for image in dataset["images"]:
        source = image.get("source_group")
        if source is not None:
            group_folds.setdefault(str(source), set()).add(
                fold_by_image[int(image["id"])]
            )
    split_groups = [source for source, values in group_folds.items() if len(values) > 1]
    if split_groups:
        raise RuntimeError(
            "Cross-validation split source variants across folds: "
            + ", ".join(sorted(split_groups)[:5])
        )


def _write_fold_annotations(
    dataset: dict[str, Any], validation_ids: list[int], work_dir: Path, fold: int
) -> dict[str, Path]:
    validation_set = set(validation_ids)
    image_by_id = {int(image["id"]): image for image in dataset["images"]}
    train_ids = set(image_by_id) - validation_set
    if not train_ids or not validation_set:
        raise ValueError(f"Fold {fold} has an empty train or validation partition.")
    output_dir = work_dir / "folds" / f"fold_{fold}" / "coco"
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = {"train": output_dir / "train.json", "valid": output_dir / "valid.json"}
    for split, image_ids in (("train", train_ids), ("valid", validation_set)):
        payload = {
            "images": [image_by_id[image_id] for image_id in sorted(image_ids)],
            "annotations": [
                annotation for annotation in dataset["annotations"]
                if int(annotation["image_id"]) in image_ids
            ],
            "categories": dataset["categories"],
        }
        if not payload["annotations"]:
            raise ValueError(f"Fold {fold} {split} has no instance annotations.")
        paths[split].write_text(json.dumps(payload), encoding="utf-8")
    return paths


def _fold_signature(
    session: str,
    model_name: str,
    fold: int,
    args: argparse.Namespace,
    annotations: dict[str, Path],
) -> tuple[str, dict[str, Any]]:
    signature_data = {
        "session": session,
        "model": model_name,
        "fold": fold,
        "seed": args.seed + fold,
        "epochs": args.epochs,
        "batch_size": args.batch_size,
        "workers": args.workers,
        "gpus": args.gpus,
        "image_size": args.image_size,
        "train_json": hashlib.sha256(annotations["train"].read_bytes()).hexdigest(),
        "valid_json": hashlib.sha256(annotations["valid"].read_bytes()).hexdigest(),
    }
    signature = hashlib.sha256(
        json.dumps(signature_data, sort_keys=True).encode("utf-8")
    ).hexdigest()
    return signature, signature_data


def _config_path(model: str) -> Path:
    if not (MMDET_SOURCE / "tools" / "train.py").is_file():
        if MMDET_SOURCE.exists():
            shutil.rmtree(MMDET_SOURCE)
        _run([
            "git", "clone", "--depth", "1", "--branch", "v2.28.2",
            "https://github.com/open-mmlab/mmdetection.git", str(MMDET_SOURCE),
        ])
    config = {
        "yolact": MMDET_SOURCE / "configs/yolact/yolact_r50_1x8_coco.py",
        "solov2": MMDET_SOURCE / "configs/solov2/solov2_r50_fpn_1x_coco.py",
    }[model]
    if not config.is_file():
        raise FileNotFoundError(f"Expected MMDetection config missing: {config}")
    return config


def _set_num_classes(node: Any, count: int) -> None:
    if isinstance(node, dict):
        for key, value in node.items():
            if key == "num_classes":
                node[key] = count
            else:
                _set_num_classes(value, count)
    elif isinstance(node, (list, tuple)):
        for item in node:
            _set_num_classes(item, count)


def _update_resize(node: Any, size: int) -> None:
    if isinstance(node, dict):
        for key, value in node.items():
            if key == "img_scale":
                node[key] = (size, size)
            else:
                _update_resize(value, size)
    elif isinstance(node, list):
        for item in node:
            _update_resize(item, size)


def _build_config(
    model_name: str,
    classes: list[str],
    annotations: dict[str, Path],
    work_dir: Path,
    args: argparse.Namespace,
) -> Path:
    from mmcv import Config

    config_path = _config_path(model_name)
    cfg = Config.fromfile(str(config_path))
    cfg.model = cfg.model.copy()
    _set_num_classes(cfg.model, len(classes))

    train_pipeline = cfg.data.train.pipeline
    test_pipeline = cfg.data.val.pipeline
    if model_name == "solov2":
        _update_resize(train_pipeline, args.image_size)
        _update_resize(test_pipeline, args.image_size)
    else:
        for transform in train_pipeline:
            if transform.get("type") == "Resize":
                transform["img_scale"] = (args.image_size, args.image_size)
        for aug in test_pipeline:
            if aug.get("type") == "MultiScaleFlipAug":
                aug["img_scale"] = (args.image_size, args.image_size)
    cfg.data = dict(
        samples_per_gpu=args.batch_size,
        workers_per_gpu=args.workers,
        train=dict(
            type="CocoDataset", ann_file=str(annotations["train"]),
            img_prefix=str(work_dir / "images" / "train") + os.sep,
            classes=tuple(classes), pipeline=train_pipeline,
        ),
        val=dict(
            type="CocoDataset", ann_file=str(annotations["valid"]),
            img_prefix=str(work_dir / "images" / "valid") + os.sep,
            classes=tuple(classes), pipeline=test_pipeline,
            test_mode=True,
        ),
        test=dict(
            type="CocoDataset", ann_file=str(annotations["test"]),
            img_prefix=str(work_dir / "images" / "test") + os.sep,
            classes=tuple(classes), pipeline=test_pipeline,
            test_mode=True,
        ),
    )
    cfg.optimizer.lr = (1e-3 if model_name == "yolact" else 1e-2) * (
        args.batch_size * args.gpus / 8
    )
    cfg.lr_config = dict(
        policy="step", warmup="linear", warmup_iters=500, warmup_ratio=0.1,
        step=[max(1, int(args.epochs * 0.70)), max(2, int(args.epochs * 0.90))],
    )
    cfg.runner = dict(type="EpochBasedRunner", max_epochs=args.epochs)
    cfg.checkpoint_config = dict(
        interval=max(1, args.epochs // 4), max_keep_ckpts=2, save_last=True,
    )
    cfg.evaluation = dict(
        interval=args.epochs, metric=["bbox", "segm"],
        classwise=True, save_best="segm_mAP",
    )
    cfg.log_config.interval = 50
    cfg.work_dir = str(work_dir)
    cfg.seed = args.seed
    cfg.gpu_ids = list(range(args.gpus))
    cfg.device = "cuda"
    cfg.dist_params = dict(backend="nccl")
    cfg.resume_from = str(work_dir / "latest.pth") if args.resume else None
    cfg.load_from = None
    if model_name == "yolact":
        cfg.img_size = args.image_size
        cfg.model["bbox_head"]["anchor_generator"]["strides"] = [
            args.image_size / value for value in (69, 35, 18, 9, 5)
        ]
        cfg.model["bbox_head"]["anchor_generator"]["centers"] = [
            (args.image_size * 0.5 / value, args.image_size * 0.5 / value)
            for value in (69, 35, 18, 9, 5)
        ]

    config_out = work_dir / "effective_config.py"
    config_out.parent.mkdir(parents=True, exist_ok=True)
    cfg.dump(str(config_out))
    return config_out


def _segmentation_ap50_by_class(
    config_path: Path, predictions_path: Path
) -> dict[str, Any]:
    import numpy as np
    from mmcv import Config, load
    from mmdet.datasets import build_dataset
    from pycocotools.cocoeval import COCOeval

    cfg = Config.fromfile(str(config_path))
    dataset = build_dataset(cfg.data.test)
    results = load(str(predictions_path))
    result_files = dataset.results2json(
        results, str(predictions_path.with_suffix(""))
    )
    coco_eval = COCOeval(
        dataset.coco,
        dataset.coco.loadRes(result_files["segm"]),
        iouType="segm",
    )
    coco_eval.params.catIds = dataset.cat_ids
    coco_eval.params.imgIds = dataset.img_ids
    coco_eval.params.maxDets = [100, 300, 1000]
    coco_eval.params.iouThrs = np.asarray([0.5])
    coco_eval.evaluate()
    coco_eval.accumulate()
    precision = coco_eval.eval["precision"][0, :, :, 0, -1]
    classwise: dict[str, float | None] = {}
    for class_index, category_id in enumerate(dataset.cat_ids):
        values = precision[:, class_index]
        values = values[values > -1]
        classwise[dataset.CLASSES[class_index]] = (
            float(values.mean()) if values.size else None
        )
    valid_scores = [score for score in classwise.values() if score is not None]
    return {
        "segm_mAP50": float(np.mean(valid_scores)) if valid_scores else None,
        "per_class_segm_mAP50": classwise,
    }


def run_session(model_name: str, session: str, argv: list[str] | None = None) -> int:
    arguments = _filter_notebook_args(
        list(sys.argv[1:] if argv is None else argv)
    )
    entrypoint = Path(globals().get("__file__", sys.argv[0])).resolve()
    bootstrap_result = _bootstrap(entrypoint, arguments, session)
    if bootstrap_result is not None:
        return bootstrap_result
    runtime_python = Path(sys.executable)
    _install_environment(runtime_python)

    parser = argparse.ArgumentParser(
        description=f"{session}: {model_name.upper()} instance segmentation on Kaggle"
    )
    parser.add_argument(
        "--data-root", default=None,
        help="Dataset directory or data.yaml; auto-detects SkinDisNet_2_Annotation on Kaggle.",
    )
    parser.add_argument("--work-dir", default=None)
    parser.add_argument(
        "--epochs", type=int, default=12,
        help="Training epochs per fold; the run prints an estimate for the selected folds.",
    )
    parser.add_argument("--folds", default="0,1,2,3,4",
                        help="Comma-separated folds to run, from 0 through 4.")
    parser.add_argument("--gpus", type=int, default=2,
                        help="Number of Kaggle GPUs for distributed training.")
    parser.add_argument("--batch-size", type=int, default=2)
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument(
        "--image-size", type=int, default=550 if model_name == "yolact" else 640
    )
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--resume", action="store_true")
    args = parser.parse_args(arguments)
    if args.epochs < 1 or args.batch_size < 1 or args.workers < 0 or args.gpus < 1:
        parser.error("--epochs, --batch-size, and --gpus must be positive; --workers cannot be negative")
    if args.image_size < 320:
        parser.error("--image-size must be at least 320")
    try:
        selected_folds = sorted({
            int(part.strip()) for part in args.folds.split(",") if part.strip()
        })
    except ValueError:
        parser.error("--folds must be comma-separated integers from 0 through 4")
    if not selected_folds or any(fold not in range(5) for fold in selected_folds):
        parser.error("--folds must contain one or more comma-separated values from 0 through 4")

    import torch

    if not torch.cuda.is_available() or torch.cuda.device_count() < args.gpus:
        visible = torch.cuda.device_count() if torch.cuda.is_available() else 0
        raise RuntimeError(
            f"Requested {args.gpus} GPU(s), but only {visible} are visible. "
            "Enable Kaggle T4 x2 or lower --gpus."
        )
    print(
        f"[gpu] DDP on {args.gpus} device(s): "
        + ", ".join(torch.cuda.get_device_name(index) for index in range(args.gpus)),
        flush=True,
    )

    dataset_root = _find_data_root(args.data_root)
    dataset_dir = dataset_root.parent if dataset_root.is_file() else dataset_root
    is_skin_disease_yolo = (
        dataset_dir.name.lower().replace("-", "_") == "skin_disease_yolo"
    )
    has_source_manifest = (dataset_dir / "manifest.csv").is_file()
    if is_skin_disease_yolo:
        run_name = f"{session.lower()}_skin_disease_yolo"
    elif has_source_manifest:
        run_name = f"{session.lower()}_skindisnet2"
    else:
        run_name = session.lower()
    work_dir = Path(args.work_dir or f"/kaggle/working/{run_name}")
    work_dir.mkdir(parents=True, exist_ok=True)
    classes, annotations = prepare_dataset(
        dataset_root, work_dir, pool_all_splits=is_skin_disease_yolo
    )
    source_group_count = _assert_test_source_isolation(
        annotations, skip=is_skin_disease_yolo
    )
    development = _make_development_coco(
        annotations, work_dir / "coco" / "development.json",
        work_dir / "images",
        pool_test=is_skin_disease_yolo,
    )
    folds = _stratified_folds(development, 5, args.seed)
    _assert_fold_source_isolation(development, folds)
    test_note = (
        "all splits pooled (test NOT independent)"
        if is_skin_disease_yolo
        else "test split locked"
    )
    print(
        f"[cv] {len(development['images'])} development images, "
        f"{source_group_count} source groups, fold sizes="
        f"{[len(fold) for fold in folds]}, {test_note}",
        flush=True,
    )
    summary_path = dataset_dir / "summary.json"
    dataset_evaluation_note = None
    if is_skin_disease_yolo:
        dataset_evaluation_note = (
            "skin_disease_yolo (Roboflow, 19 classes): source groups from .rf. filename "
            "stems. All splits pooled because source variants span train/test (51 groups "
            "overlap with test). The original test split is NOT an independent test set. "
            "Each source group is held out as a unit across all 5 folds."
        )
    elif summary_path.is_file():
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        dataset_evaluation_note = summary.get("evaluation_note")
    if dataset_evaluation_note:
        print(f"[evaluation note] {dataset_evaluation_note}", flush=True)
    total_iterations = 0
    for fold in selected_folds:
        train_images = len(development["images"]) - len(folds[fold])
        total_iterations += args.epochs * (
            (train_images + args.gpus * args.batch_size - 1)
            // (args.gpus * args.batch_size)
        )
    print(
        f"[training] {model_name.upper()} | folds={selected_folds} | "
        f"classes={len(classes)} | epochs/fold={args.epochs} | "
        f"batch/GPU={args.batch_size} | GPUs={args.gpus} | "
        f"estimated_total_iterations={total_iterations}",
        flush=True,
    )

    env = {**os.environ, "PYTHONUNBUFFERED": "1", "MPLBACKEND": "Agg"}
    started = time.time()
    for fold in selected_folds:
        fold_dir = work_dir / "folds" / f"fold_{fold}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        fold_annotations = _write_fold_annotations(
            development, folds[fold], work_dir, fold
        )
        fold_annotations["test"] = fold_annotations["valid"]
        signature, signature_data = _fold_signature(
            session, model_name, fold, args, fold_annotations
        )
        complete_path = fold_dir / "complete.json"
        if complete_path.is_file():
            existing = json.loads(complete_path.read_text(encoding="utf-8"))
            if (
                existing.get("signature") == signature
                and Path(existing.get("checkpoint", "")).is_file()
                and Path(existing.get("oof_metrics_log", "")).is_file()
            ):
                print(f"[fold {fold}] already complete; reusing saved result", flush=True)
                continue
        signature_path = fold_dir / "run_signature.json"
        latest_checkpoint = fold_dir / "latest.pth"
        if args.resume:
            if not latest_checkpoint.is_file():
                raise FileNotFoundError(
                    f"--resume requested but fold {fold} has no checkpoint: "
                    f"{latest_checkpoint}"
                )
            if not signature_path.is_file():
                raise RuntimeError(
                    f"Cannot safely resume fold {fold}: {signature_path} is missing."
                )
            saved_signature = json.loads(
                signature_path.read_text(encoding="utf-8")
            )
            if saved_signature.get("signature") != signature:
                raise RuntimeError(
                    f"Cannot resume fold {fold}: its saved run settings or "
                    "annotations differ from this run. Start without --resume "
                    "to train that fold from scratch."
                )
        else:
            signature_path.write_text(
                json.dumps(
                    {**signature_data, "signature": signature}, indent=2
                ),
                encoding="utf-8",
            )
        fold_args = argparse.Namespace(**vars(args))
        fold_args.resume = args.resume
        config = _build_config(
            model_name, classes, fold_annotations, fold_dir, fold_args
        )
        train_log = fold_dir / "train.log"
        train_command = [
            str(runtime_python), "-m", "torch.distributed.launch",
            "--nproc_per_node", str(args.gpus),
            "--master_port", str(20000 + (os.getpid() + fold) % 40000),
            str(MMDET_SOURCE / "tools/train.py"), str(config),
            "--launcher", "pytorch", "--seed", str(args.seed + fold),
        ]
        print(f"[fold {fold}] training; log={train_log}", flush=True)
        with train_log.open("a", encoding="utf-8") as log:
            subprocess.run(
                train_command, check=True, env=env, stdout=log,
                stderr=subprocess.STDOUT,
            )
        candidates = sorted(fold_dir.glob("best_segm_mAP_epoch_*.pth"))
        if not candidates:
            candidates = sorted(fold_dir.glob("epoch_*.pth"))
        if not candidates and (fold_dir / "latest.pth").is_file():
            candidates = [fold_dir / "latest.pth"]
        if not candidates:
            raise FileNotFoundError(f"Fold {fold} produced no checkpoint in {fold_dir}")
        checkpoint = max(candidates, key=lambda path: path.stat().st_mtime)
        test_log = fold_dir / "oof_metrics.log"
        predictions_path = fold_dir / "oof_predictions.pkl"
        test_command = [
            str(runtime_python), str(MMDET_SOURCE / "tools/test.py"),
            str(config), str(checkpoint), "--out", str(predictions_path),
            "--eval", "bbox", "segm",
            "--eval-options", "classwise=True",
        ]
        with test_log.open("w", encoding="utf-8") as log:
            subprocess.run(
                test_command, check=True, env=env, stdout=log,
                stderr=subprocess.STDOUT,
            )
        ap50 = _segmentation_ap50_by_class(config, predictions_path)
        predictions_path.unlink()
        fold_report = {
            "fold": fold,
            "signature": signature,
            "seed": args.seed + fold,
            "train_images": len(development["images"]) - len(folds[fold]),
            "validation_images": len(folds[fold]),
            "checkpoint": str(checkpoint),
            "training_log": str(train_log),
            "oof_metrics_log": str(test_log),
            **ap50,
        }
        complete_path.write_text(
            json.dumps(fold_report, indent=2), encoding="utf-8"
        )
        print(
            f"[fold {fold}] complete; segm mAP50={ap50['segm_mAP50']}; "
            f"per-class results={ap50['per_class_segm_mAP50']}; log={test_log}",
            flush=True,
        )

    all_fold_reports: list[dict[str, Any]] = []
    for fold in range(5):
        fold_dir = work_dir / "folds" / f"fold_{fold}"
        complete_path = fold_dir / "complete.json"
        signature_path = fold_dir / "run_signature.json"
        if not complete_path.is_file() or not signature_path.is_file():
            continue
        fold_annotations = _write_fold_annotations(
            development, folds[fold], work_dir, fold
        )
        fold_annotations["test"] = fold_annotations["valid"]
        expected_signature, _ = _fold_signature(
            session, model_name, fold, args, fold_annotations
        )
        completed = json.loads(complete_path.read_text(encoding="utf-8"))
        saved_signature = json.loads(signature_path.read_text(encoding="utf-8"))
        if (
            completed.get("signature") == expected_signature
            and saved_signature.get("signature") == expected_signature
            and Path(completed.get("checkpoint", "")).is_file()
            and Path(completed.get("oof_metrics_log", "")).is_file()
        ):
            all_fold_reports.append(completed)

    fold_map50s = [
        result["segm_mAP50"] for result in all_fold_reports
        if result.get("segm_mAP50") is not None
    ]
    per_class_fold_scores = {
        class_name: [
            result["per_class_segm_mAP50"][class_name]
            for result in all_fold_reports
            if result.get("per_class_segm_mAP50", {}).get(class_name) is not None
        ]
        for class_name in classes
    }
    per_class_cv = {
        class_name: {
            "mean_segm_mAP50": statistics.fmean(scores) if scores else None,
            "folds_with_class": len(scores),
        }
        for class_name, scores in per_class_fold_scores.items()
    }
    report = {
        "session": session,
        "model": model_name.upper(),
        "dataset_root": str(dataset_root),
        "classes": classes,
        "source_group_count": source_group_count,
        "source_grouped_cross_validation": bool(source_group_count),
        "dataset_evaluation_note": dataset_evaluation_note,
        "epochs": args.epochs,
        "batch_size": args.batch_size,
        "gpus": args.gpus,
        "image_size": args.image_size,
        "selected_folds": selected_folds,
        "completed_folds": [result["fold"] for result in all_fold_reports],
        "cross_validation_complete": len(all_fold_reports) == 5,
        "total_iterations_estimate": total_iterations,
        "mean_fold_segm_mAP50": (
            statistics.fmean(fold_map50s) if fold_map50s else None
        ),
        "std_fold_segm_mAP50": (
            statistics.pstdev(fold_map50s) if fold_map50s else None
        ),
        "per_class_mean_fold_segm_mAP50": per_class_cv,
        "fold_results": all_fold_reports,
        "elapsed_seconds": round(time.time() - started, 2),
        "framework": "MMDetection 2.28.2 / MMCV-Full 1.7.2",
        "evaluation": (
            "Each fold's held-out validation split is evaluated out-of-fold. "
            + (
                "All splits pooled (skin_disease_yolo); original test is NOT independent. "
                if is_skin_disease_yolo else
                "The supplied test split is not used during cross-validation. "
            )
            + "Source groups are kept in a single fold when available."
        ),
    }
    result_path = work_dir / "results.json"
    result_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(f"[complete] fold reports: {len(all_fold_reports)}", flush=True)
    print(f"[complete] run summary: {result_path}", flush=True)
    return 0


if __name__ == "__main__":
    exit_code = run_session('solov2', 'SESSION_33')
    if "ipykernel" in sys.modules:
        if exit_code != 0:
            raise RuntimeError(
                f"SESSION_33 failed with exit code {exit_code}."
            )
        print("[complete] SESSION_33 finished successfully.", flush=True)
    else:
        raise SystemExit(exit_code)



[args] ignored Jupyter kernel arguments: ['-f', '/tmp/tmpu59dz2zw.json', '--HistoryManager.hist_file=:memory:']
+ /usr/bin/python3 -m uv python install 3.10


+ /usr/bin/python3 -m uv venv --seed --python 3.10 /kaggle/working/session32_33_mmdet_py310


 Downloaded cpython-3.10.22-linux-x86_64-gnu (download)
Installed Python 3.10.22 in 1.29s
 + cpython-3.10.22-linux-x86_64-gnu (python3.10)
Using CPython 3.10.22
Creating virtual environment with seed packages at: session32_33_mmdet_py310


[environment] installing the pinned CUDA 11.3 stack
+ /kaggle/working/session32_33_mmdet_py310/bin/python -m pip install -q --upgrade --force-reinstall --no-cache-dir setuptools==69.5.1 wheel==0.44.0


         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
 + packaging==26.3
 + pip==26.2.2
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source session32_33_mmdet_py310/bin/activate


+ /kaggle/working/session32_33_mmdet_py310/bin/python -m pip install -q --upgrade --no-cache-dir opencv-python==4.8.1.78
+ /kaggle/working/session32_33_mmdet_py310/bin/python -m pip install -q numpy==1.23.5
+ /kaggle/working/session32_33_mmdet_py310/bin/python -m pip install -q torch==1.12.0+cu113 torchvision==0.13.0+cu113 --index-url https://download.pytorch.org/whl/cu113 --extra-index-url https://pypi.org/simple
+ /kaggle/working/session32_33_mmdet_py310/bin/python -m pip install -q --no-deps --find-links https://download.openmmlab.com/mmcv/dist/cu113/torch1.12.0/index.html mmcv-full==1.7.2
+ /kaggle/working/session32_33_mmdet_py310/bin/python -m pip install -q --no-deps mmdet==2.28.2
+ /kaggle/working/session32_33_mmdet_py310/bin/python -m pip install -q addict matplotlib==3.7.5 opencv-python==4.8.1.78 packaging Pillow pycocotools PyYAML rich scipy==1.10.1 six terminaltables tqdm yapf==0.40.1
[environment] verified: 1.12.0+cu113 0.13.0+cu113 1.7.2 2.28.2 setuptools=69.5.1
[python] r

/kaggle/working/session32_33_mmdet_py310/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
Cloning into '/kaggle/working/mmdetection_v2.28.2'...


+ git clone --depth 1 --branch v2.28.2 https://github.com/open-mmlab/mmdetection.git /kaggle/working/mmdetection_v2.28.2


Note: switching to 'e9cae2d0787cd5c2fc6165a6061f92fa09e48fb1'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false



[fold 0] training; log=/kaggle/working/session_33_skin_disease_yolo/folds/fold_0/train.log
loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.37s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=2.49s).
Accumulating evaluation results...
DONE (t=0.56s).
[fold 0] complete; segm mAP50=0.15081704589016573; per-class results={'acne': 0.037156035948057935, 'Darier_s-Disease': 0.1271564729057583, 'Epidermolysis-Bullosa-Pruriginosa': 0.016536568984053545, 'Hailey-Hailey-Disease': 0.044023264808964915, 'Hemangiome': 0.16246965490771975, 'Impetigo': 0.03459800444588223, 'Leishmanios': 0.08896918532710449, 'Lupus-Erythematosus-Chronicus-Discoides': 0.022487813277918267, 'Melanoma': 0.38350749839670323, 'Molluscum-Contagiosum': 0.04934090518506704, 'Porokeratosis': 0.04958848911578698, 'Psoriasis': 0.042881473409624685, 'Tinea-Corporis': 0.10723406307659152, 'Tungias